In [2]:
import numpy as np
from review_data import make_reviews              # the 60 reviews from last class
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [12]:


# each word = 3 numbers we chose:  [ royalty, male-ness, female-ness ]
words = {
    "king":  np.array([0.9, 0.9, 0.1]),
    "queen": np.array([0.9, 0.1, 0.9]),
    "man":   np.array([0.1, 0.9, 0.1]),
    "woman": np.array([0.1, 0.1, 0.9]),
}

def cosine(a, b):
    """1 = same direction (same meaning), 0 = unrelated."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(cosine(words["king"], words["queen"]))   # 0.607  (both royal)
print(cosine(words["king"], words["woman"]))   # 0.232  (little in common)

0.6073619631901841
0.232129809914724


In [13]:



texts, labels = make_reviews()

lsa = make_pipeline(
    TfidfVectorizer(),                              # wide sparse table (Day 23)
    TruncatedSVD(n_components=20, random_state=0),  # squeeze to 20 numbers
    Normalizer(),                                   # so cosine = a simple dot
)
embeddings = lsa.fit_transform(texts)             # 60 reviews x 20 numbers


/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat


In [16]:
texts

['the food was good and the service was fast',
 'good food and very friendly staff',
 'great service and great value for money',
 'the momo was delicious and the staff were friendly',
 'delicious food, fast service, good price',
 'excellent food and excellent service',
 'the curry was delicious and the staff was kind',
 'friendly staff and fresh food',
 'fresh ingredients and good flavour',
 'the dal bhat was delicious and hot',
 'good value and the service was quick',
 'quick service and delicious coffee',
 'the staff was friendly and the food was fresh',
 'great flavour and a clean place',
 'clean tables and very good food',
 'the thukpa was excellent and hot',
 'excellent value, the food was fresh',
 'we loved the food, service was fast',
 'the biryani was delicious and the portion was generous',
 'generous portion and good price',
 'the tea was good and the staff smiled',
 'friendly waiter and delicious pastries',
 'the food arrived hot and fresh',
 'hot food, fast service, friendl

In [15]:
embeddings

array([[ 0.80743957,  0.12914738, -0.10508404, ...,  0.03164736,
         0.12607563, -0.051387  ],
       [ 0.41985648, -0.59300725,  0.35656824, ...,  0.02317347,
        -0.10688793, -0.14957853],
       [ 0.18187721, -0.18189302, -0.31352559, ..., -0.06209056,
         0.09867776,  0.09628734],
       ...,
       [ 0.77813601,  0.24762089, -0.10512282, ...,  0.00688135,
         0.13571798, -0.08047416],
       [ 0.28076087, -0.36539804,  0.4047704 , ..., -0.23979179,
        -0.03284955, -0.03663658],
       [ 0.15159508, -0.35355498, -0.59114873, ..., -0.06206411,
        -0.1455494 , -0.03607674]], shape=(60, 20))

In [10]:
svd_model  = lsa.named_steps['truncatedsvd']

# Get the list of all words (this is just a giant list of strings)
all_words = svd_model.components_

In [11]:
all_words

array([[ 0.00565316,  0.28248634,  0.03728817, ...,  0.04365335,
         0.0492841 ,  0.00565316],
       [-0.043067  , -0.20300542, -0.00336599, ..., -0.03317208,
         0.0287851 , -0.043067  ],
       [ 0.01518802, -0.09580961,  0.01329784, ...,  0.00871124,
         0.06672831,  0.01518802],
       ...,
       [-0.03872764, -0.01039706,  0.07913025, ..., -0.05878593,
         0.24531935, -0.03872764],
       [ 0.04535398,  0.04101291,  0.15154879, ...,  0.04266579,
         0.04855402,  0.04535398],
       [ 0.04797895, -0.01658413,  0.37782722, ...,  0.04658456,
         0.12528739,  0.04797895]], shape=(20, 69))

In [20]:
q = lsa.transform(["the staff were rude and the food was cold"])[0]
best = np.argsort(-(embeddings @ q))[:3]

for i in best: print(texts[i])

the momo was cold and the staff were rude
the staff was rude and the food was stale
the curry was bland and the staff was rude


/var/folders/kp/v0h7kc1d16x3671769czcdnc0000gn/T/ipykernel_69357/3992443101.py:2: RuntimeWarning: divide by zero encountered in matmul
  best = np.argsort(-(embeddings @ q))[:3]
/var/folders/kp/v0h7kc1d16x3671769czcdnc0000gn/T/ipykernel_69357/3992443101.py:2: RuntimeWarning: overflow encountered in matmul
  best = np.argsort(-(embeddings @ q))[:3]
/var/folders/kp/v0h7kc1d16x3671769czcdnc0000gn/T/ipykernel_69357/3992443101.py:2: RuntimeWarning: invalid value encountered in matmul
  best = np.argsort(-(embeddings @ q))[:3]
